<a href="https://colab.research.google.com/github/mrxy56/RAG-based-Twitter-Sentiment-Analysis/blob/main/Twitter_Sentiment_Analysis_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [47]:
#!pip install -q flask flask-cors requests nltk ibm-watson chromadb sentence-transformers transformers accelerate

In [42]:
import re
import os
import requests
import nltk
import numpy as np
import threading
import torch
import chromadb

from collections import Counter

from flask import Flask, request, jsonify
from flask_cors import CORS

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

from ibm_cloud_sdk_core.authenticators import IAMAuthenticator

from ibm_watson import NaturalLanguageUnderstandingV1

from ibm_watson.natural_language_understanding_v1 import (
    Features,
    SentimentOptions
)

In [46]:
X_BEARER_TOKEN='AAAAAAAAAAAAAAAAAAAAAO0%2B7wEAAAAA5R4iz41hmfkMd5sgXJaVozz3Wbg%3DpT4HNyET4cR089o3JKGEm8Mb16hQ65fpkRM4ahmM9IBHwD3k0b'
IBM_NLU_APIKEY='A913Zj9I3s8MIo5t6THWq3lPPfLd9YRGnoWHYrXuKfec'
IBM_NLU_URL='https://api.eu-de.natural-language-understanding.watson.cloud.ibm.com/instances/3dae53de-ffbe-4dad-8890-706f7713858b'
IBM_NLU_VERSION='2022-04-07'

In [4]:
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

EN_STOPWORDS = set(stopwords.words("english"))

In [48]:
authenticator = IAMAuthenticator(IBM_NLU_APIKEY)

nlu = NaturalLanguageUnderstandingV1(
    version="2022-04-07",
    authenticator=authenticator
)

nlu.set_service_url(IBM_NLU_URL)

In [6]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [49]:
chroma_client = chromadb.PersistentClient(
    path="/content/twitter_chroma"
)

collection = chroma_client.get_or_create_collection(
    name="twitter_rag",
    metadata={"hnsw:space": "cosine"}
)

print("Vector database size:", collection.count())

Vector database size: 0


In [50]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()

print("Qwen model loaded.")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen model loaded.


In [51]:
def clean_text(text):
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#(\w+)", r"\1", text)
    text = re.sub(r"[^A-Za-z0-9\s']", " ", text)

    return re.sub(
        r"\s+",
        " ",
        text
    ).lower().strip()

In [52]:
def tokens_from_text(text):
    tokens = word_tokenize(
        clean_text(text)
    )

    return [
        token
        for token in tokens
        if (
            token.isalpha()
            and token not in EN_STOPWORDS
            and len(token) > 2
        )
    ]

In [53]:
def watson_sentiment(text):
    try:
        result = nlu.analyze(
            text=text,
            language="en",
            features=Features(
                sentiment=SentimentOptions()
            )
        ).get_result()

        sentiment = result[
            "sentiment"
        ]["document"]

        return {
            "label": sentiment["label"],
            "score": float(sentiment["score"])
        }

    except Exception as e:
        print("Watson error:", e)

        return {
            "label": "neutral",
            "score": 0.0
        }

In [54]:
def search_tweets(query, max_results=10):
    url = (
        "https://api.twitter.com/2/"
        "tweets/search/recent"
    )

    headers = {
        "Authorization":
        f"Bearer {X_BEARER_TOKEN}"
    }

    params = {
        "query": (
            f"({query}) "
            "-is:retweet lang:en"
        ),
        "max_results": max(
            10,
            min(int(max_results), 100)
        ),
        "tweet.fields": (
            "created_at,lang,"
            "public_metrics"
        )
    }

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=30
    )

    if response.status_code != 200:
        raise RuntimeError(response.text)

    return response.json().get(
        "data",
        []
    )

In [55]:
def analyze_tweets(tweets):
    results = []

    for tweet in tweets:
        sentiment = watson_sentiment(
            tweet["text"]
        )

        results.append({
            "id": str(tweet["id"]),
            "text": tweet["text"],
            "created_at": tweet.get(
                "created_at",
                ""
            ),
            "clean_text": clean_text(
                tweet["text"]
            ),
            "tokens": tokens_from_text(
                tweet["text"]
            ),
            "sentiment": sentiment["label"],
            "sentiment_score": sentiment["score"]
        })

    return results

In [56]:
def summarize(tweets):
    sentiment_counts = Counter(
        tweet["sentiment"]
        for tweet in tweets
    )

    token_counts = Counter()

    for tweet in tweets:
        token_counts.update(
            tweet["tokens"]
        )

    return {
        "total": len(tweets),
        "positive": sentiment_counts[
            "positive"
        ],
        "neutral": sentiment_counts[
            "neutral"
        ],
        "negative": sentiment_counts[
            "negative"
        ],
        "top_words": token_counts.most_common(
            10
        )
    }

In [57]:
def store_tweets(tweets, topic):
    if not tweets:
        return 0

    texts = [
        tweet["clean_text"]
        or tweet["text"]
        for tweet in tweets
    ]

    embeddings = embedding_model.encode(
        texts,
        normalize_embeddings=True
    ).tolist()

    collection.upsert(
        ids=[
            tweet["id"]
            for tweet in tweets
        ],
        documents=[
            tweet["text"]
            for tweet in tweets
        ],
        embeddings=embeddings,
        metadatas=[
            {
                "topic": topic,
                "sentiment":
                    tweet["sentiment"],
                "sentiment_score":
                    tweet["sentiment_score"],
                "created_at":
                    tweet["created_at"]
            }
            for tweet in tweets
        ]
    )

    return len(tweets)

In [58]:
def retrieve_tweets(
    question,
    top_k=5,
    sentiment=None,
    topic=None
):
    if collection.count() == 0:
        return []

    query_embedding = embedding_model.encode(
        question,
        normalize_embeddings=True
    ).tolist()

    filters = []

    if sentiment:
        filters.append({
            "sentiment": sentiment
        })

    if topic:
        filters.append({
            "topic": topic
        })

    where = None

    if len(filters) == 1:
        where = filters[0]

    elif len(filters) > 1:
        where = {
            "$and": filters
        }

    arguments = {
        "query_embeddings": [
            query_embedding
        ],
        "n_results": min(
            int(top_k),
            collection.count()
        )
    }

    if where:
        arguments["where"] = where

    result = collection.query(
        **arguments
    )

    items = []

    for tweet_id, text, metadata, distance in zip(
        result["ids"][0],
        result["documents"][0],
        result["metadatas"][0],
        result["distances"][0]
    ):
        items.append({
            "id": tweet_id,
            "text": text,
            "sentiment": metadata[
                "sentiment"
            ],
            "sentiment_score": metadata[
                "sentiment_score"
            ],
            "topic": metadata["topic"],
            "similarity": 1 - float(distance)
        })

    return items

In [59]:
def generate_answer(question, sources):
    context = "\n\n".join([
        (
            f"[Source {index}]\n"
            f"Tweet: {item['text']}\n"
            f"Sentiment: {item['sentiment']}"
        )
        for index, item in enumerate(
            sources,
            start=1
        )
    ])

    messages = [
        {
            "role": "system",
            "content": (
                "You are a Twitter data "
                "analysis assistant. "
                "Answer only from the given "
                "tweets. Do not invent facts. "
                "Cite sources using "
                "[Source 1], [Source 2], etc."
            )
        },
        {
            "role": "user",
            "content": f"""
Question:
{question}

Tweets:
{context}

Answer the question briefly.
"""
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = output[
        0,
        inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

In [60]:
def answer_question(
    question,
    top_k=5,
    sentiment=None,
    topic=None
):
    sources = retrieve_tweets(
        question=question,
        top_k=top_k,
        sentiment=sentiment,
        topic=topic
    )

    if not sources:
        return {
            "answer":
                "No relevant tweets were found.",
            "sources": []
        }

    answer = generate_answer(
        question,
        sources
    )

    return {
        "question": question,
        "answer": answer,
        "sources": sources
    }

In [61]:
LABELS = [
    "positive",
    "neutral",
    "negative"
]

In [62]:
def evaluate_sentiment(items):
    matrix = np.zeros(
        (3, 3),
        dtype=int
    )

    for item in items:
        true_label = item["true_label"]

        predicted = watson_sentiment(
            item["text"]
        )["label"]

        if (
            true_label in LABELS
            and predicted in LABELS
        ):
            true_index = LABELS.index(
                true_label
            )

            predicted_index = LABELS.index(
                predicted
            )

            matrix[
                true_index,
                predicted_index
            ] += 1

    total = matrix.sum()
    accuracy = (
        np.trace(matrix) / total
        if total
        else 0
    )

    per_class = {}
    f1_scores = []

    for index, label in enumerate(LABELS):
        tp = matrix[index, index]
        fp = matrix[:, index].sum() - tp
        fn = matrix[index, :].sum() - tp

        precision = (
            tp / (tp + fp)
            if tp + fp
            else 0
        )

        recall = (
            tp / (tp + fn)
            if tp + fn
            else 0
        )

        f1 = (
            2 * precision * recall
            / (precision + recall)
            if precision + recall
            else 0
        )

        f1_scores.append(f1)

        per_class[label] = {
            "precision": precision,
            "recall": recall,
            "f1": f1
        }

    return {
        "labels": LABELS,
        "confusion_matrix":
            matrix.tolist(),
        "accuracy": float(accuracy),
        "macro_f1": float(
            np.mean(f1_scores)
        ),
        "per_class": per_class
    }

In [21]:
sentiment_test_data = [
    {
        "text": "I love this product.",
        "true_label": "positive"
    },
    {
        "text": "This update is terrible.",
        "true_label": "negative"
    },
    {
        "text": "The update was released today.",
        "true_label": "neutral"
    }
]

In [63]:
def evaluate_retrieval(
    test_cases,
    top_k=5
):
    precisions = []
    recalls = []
    reciprocal_ranks = []

    details = []

    for case in test_cases:
        results = retrieve_tweets(
            question=case["question"],
            top_k=top_k,
            sentiment=case.get(
                "sentiment"
            ),
            topic=case.get("topic")
        )

        retrieved_ids = [
            result["id"]
            for result in results
        ]

        relevant_ids = set(
            case["relevant_ids"]
        )

        relevant_found = [
            tweet_id
            for tweet_id in retrieved_ids
            if tweet_id in relevant_ids
        ]

        precision = (
            len(relevant_found)
            / len(retrieved_ids)
            if retrieved_ids
            else 0
        )

        recall = (
            len(relevant_found)
            / len(relevant_ids)
            if relevant_ids
            else 0
        )

        reciprocal_rank = 0

        for rank, tweet_id in enumerate(
            retrieved_ids,
            start=1
        ):
            if tweet_id in relevant_ids:
                reciprocal_rank = 1 / rank
                break

        precisions.append(precision)
        recalls.append(recall)
        reciprocal_ranks.append(
            reciprocal_rank
        )

        details.append({
            "question": case["question"],
            "retrieved_ids": retrieved_ids,
            "relevant_ids":
                list(relevant_ids),
            f"precision@{top_k}":
                precision,
            f"recall@{top_k}":
                recall,
            "reciprocal_rank":
                reciprocal_rank
        })

    return {
        f"mean_precision@{top_k}":
            float(np.mean(precisions)),
        f"mean_recall@{top_k}":
            float(np.mean(recalls)),
        "mrr": float(
            np.mean(reciprocal_ranks)
        ),
        "details": details
    }

In [64]:
retrieval_test_data = [
    {
        "question":
            "What customer service problems are mentioned?",
        "relevant_ids": [
            "tweet_id_1",
            "tweet_id_2"
        ],
        "topic": "Tesla"
    }
]

In [65]:
app = Flask(__name__)
CORS(app)

In [66]:
@app.get("/api/status")
def api_status():
    return jsonify({
        "status": "ok",
        "vector_database_size":
            collection.count()
    })

In [67]:
@app.post("/api/analyze")
def api_analyze():
    data = request.get_json() or {}

    query = data.get(
        "query",
        ""
    ).strip()

    max_results = data.get(
        "max_results",
        10
    )

    if not query:
        return jsonify({
            "error": "query is required"
        }), 400

    try:
        raw_tweets = search_tweets(
            query,
            max_results
        )

        tweets = analyze_tweets(
            raw_tweets
        )

        stored = store_tweets(
            tweets,
            query
        )

        return jsonify({
            "query": query,
            "summary": summarize(tweets),
            "stored": stored,
            "tweets": tweets
        })

    except Exception as e:
        return jsonify({
            "error": str(e)
        }), 500

In [68]:
@app.post("/api/search")
def api_search():
    data = request.get_json() or {}

    query = data.get(
        "query",
        ""
    ).strip()

    if not query:
        return jsonify({
            "error": "query is required"
        }), 400

    results = retrieve_tweets(
        question=query,
        top_k=data.get("top_k", 5),
        sentiment=data.get(
            "sentiment"
        ),
        topic=data.get("topic")
    )

    return jsonify({
        "query": query,
        "results": results
    })

In [69]:
@app.post("/api/ask")
def api_ask():
    data = request.get_json() or {}

    question = data.get(
        "question",
        ""
    ).strip()

    if not question:
        return jsonify({
            "error": "question is required"
        }), 400

    result = answer_question(
        question=question,
        top_k=data.get("top_k", 5),
        sentiment=data.get(
            "sentiment"
        ),
        topic=data.get("topic")
    )

    return jsonify(result)

In [70]:
@app.post("/api/evaluate-sentiment")
def api_evaluate_sentiment():
    data = request.get_json() or {}

    items = data.get("items", [])

    if not items:
        return jsonify({
            "error":
                "items are required"
        }), 400

    return jsonify(
        evaluate_sentiment(items)
    )

In [71]:
@app.post("/api/evaluate-retrieval")
def api_evaluate_retrieval():
    data = request.get_json() or {}

    test_cases = data.get(
        "test_cases",
        []
    )

    if not test_cases:
        return jsonify({
            "error":
                "test_cases are required"
        }), 400

    return jsonify(
        evaluate_retrieval(
            test_cases,
            top_k=data.get(
                "top_k",
                5
            )
        )
    )

In [72]:
def run_app():
    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )

In [73]:
thread = threading.Thread(
    target=run_app,
    daemon=True
)

thread.start()

time.sleep(2)

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


In [75]:
query = "Tesla"

raw_tweets = search_tweets(
    query=query,
    max_results=10
)

tweets = analyze_tweets(raw_tweets)

summary = summarize(tweets)

stored = store_tweets(
    tweets=tweets,
    topic=query
)

analyze_result = {
    "query": query,
    "summary": summary,
    "stored": stored,
    "tweets": tweets
}

analyze_result

{'query': 'Tesla',
 'summary': {'total': 10,
  'positive': 3,
  'neutral': 3,
  'negative': 4,
  'top_words': [('tesla', 9),
   ('tow', 3),
   ('tsla', 3),
   ('price', 2),
   ('elon', 2),
   ('musk', 2),
   ('would', 2),
   ('yes', 2),
   ('new', 2),
   ('time', 2)]},
 'stored': 10,
 'tweets': [{'id': '2078493866026557877',
   'text': 'Price Prediction: Tesla Poised for 12% Rally as Profit Margins Improve - 24/7 Wall St. https://t.co/V5AQpLWJis',
   'created_at': '2026-07-18T14:55:28.000Z',
   'clean_text': 'price prediction tesla poised for 12 rally as profit margins improve 24 7 wall st',
   'tokens': ['price',
    'prediction',
    'tesla',
    'poised',
    'rally',
    'profit',
    'margins',
    'improve',
    'wall'],
   'sentiment': 'negative',
   'sentiment_score': -0.260977},
  {'id': '2078493851032125815',
   'text': 'Be honest 🧐.....\n\nIf Elon musk gives you free Tesla Car 🚗 right now, would you accept?\n\nA. Yes \nB. No https://t.co/RIxExElxQ5',
   'created_at': '2026-0

In [76]:
for tweet in analyze_result.get(
    "tweets",
    []
):
    print(
        tweet["id"],
        tweet["sentiment"],
        tweet["text"]
    )

2078493866026557877 negative Price Prediction: Tesla Poised for 12% Rally as Profit Margins Improve - 24/7 Wall St. https://t.co/V5AQpLWJis
2078493851032125815 positive Be honest 🧐.....

If Elon musk gives you free Tesla Car 🚗 right now, would you accept?

A. Yes 
B. No https://t.co/RIxExElxQ5
2078493809311207686 positive @MSandraWilson Big news! You’ve officially been selected to receive a brand new Tesla vehicle.

We can’t wait to hand over the keys! 🔑 

Kindly reply “YES” to confirm your eligibility.

Signed:
ELON MUSK
2078493782908055911 negative @DrBrianReid @Tesla He bought the adapter but his would not work, this was last year as well.
2078493773563109620 negative Tesla's LFP Batteries Hold Up Best Over Time in New Study - Not a Tesla App https://t.co/XNFNY5Drbl
2078493770283127265 neutral @KatieMiller @Tesla Mrs. Goebbels
2078493760460148840 neutral Ever wanted to change your Tesla Chrome Badge to a black one? Here is how I did it. https://t.co/mN3e3q1x4x
2078493752553783585 ne

In [77]:
search_results = retrieve_tweets(
    question="What problems are users discussing?",
    top_k=5,
    topic="Tesla"
)

search_results

[{'id': '2078493719427481985',
  'text': 'Tesla Q2 Earnings Preview: The Numbers Will Beat, The Narrative May Not (NASDAQ:TSLA) | Seeking Alpha https://t.co/QYnC7jBKOf',
  'sentiment': 'positive',
  'sentiment_score': 0.307669,
  'topic': 'Tesla',
  'similarity': 0.1276518702507019},
 {'id': '2078493773563109620',
  'text': "Tesla's LFP Batteries Hold Up Best Over Time in New Study - Not a Tesla App https://t.co/XNFNY5Drbl",
  'sentiment': 'negative',
  'sentiment_score': -0.70789,
  'topic': 'Tesla',
  'similarity': 0.11615020036697388},
 {'id': '2078493719351681166',
  'text': '$TSLA || #TSLA || #tesla \n\nAt critcal weekly support, losing it will take the price to $ 270~303 range. https://t.co/2vo6YzGHrD',
  'sentiment': 'neutral',
  'sentiment_score': 0.0,
  'topic': 'Tesla',
  'similarity': 0.06579875946044922},
 {'id': '2078493752553783585',
  'text': '@jpcaudill @CyberOdysseyUSA Mavericks can tow 2,000lbs, or 4,000lbs with the Towing Package (Model Y can tow 1,600lbs standard or

In [78]:
qa_result = answer_question(
    question="What are the main complaints in these tweets?",
    top_k=5,
    sentiment="negative",
    topic="Tesla"
)

print(qa_result["answer"])

The main complaints in these tweets are:

- Tesla's LFP Batteries hold up best over time and do not perform as expected.
- The price prediction of Tesla is inaccurate.
- Tesla owners complain about backing into SuperCharger stalls with bike racks.


In [79]:
for source in qa_result["sources"]:
    print(
        source["sentiment"],
        round(source["similarity"], 3),
        source["text"]
    )

negative 0.157 Tesla's LFP Batteries Hold Up Best Over Time in New Study - Not a Tesla App https://t.co/XNFNY5Drbl
negative 0.107 Price Prediction: Tesla Poised for 12% Rally as Profit Margins Improve - 24/7 Wall St. https://t.co/V5AQpLWJis
negative 0.104 @jpcaudill @CyberOdysseyUSA Mavericks can tow 2,000lbs, or 4,000lbs with the Towing Package (Model Y can tow 1,600lbs standard or 3,500lbs with tow package). Even still… don’t Tesla owners complain literally all the time about backing into SuperCharger stalls with bike racks??
negative -0.031 @DrBrianReid @Tesla He bought the adapter but his would not work, this was last year as well.
